In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 

In [0]:
%run /Users/gainrakesh2001@gmail.com/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","coustmer","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = "/Volumes/workspace/default/sportsbar_raw/customers/"
print(catalog,data_source)
print(base_path)

In [0]:
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load(base_path)

df = df.withColumn("read_timestamp",F.current_timestamp())
df = df.select("*","_metadata.file_name","_metadata.file_size")
display(df)

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### SILVER PROCESSING

In [0]:
df_bronze = spark.sql(f"""select * from {catalog}.{bronze_schema}.{data_source}""")
df_bronze.show()

In [0]:
#Check the duplicate
df_duplicate = df_bronze.groupBy("customer_id").count().withColumnRenamed("count", "duplicate_customer").filter(F.col("duplicate_customer") > 1)
display(df_duplicate)

In [0]:
# remove the duplicate and save the data in silver table
print("Row before duplicate dropped: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Row after duplicate dropped: ", df_silver.count())

In [0]:
# Now i will check the some data in name column that has some gap 
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
# Remove the gap from name column
df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))
display(df_silver)

In [0]:
#check after trim , remove the gap from the name column 
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver.select("city").distinct().show() # If you can see there are same city but spaling wrong , need to fix that

In [0]:
city_mapping = {
    "Bengalore" : "Bengaluru",
    "Bengaluruu" : "Bengaluru",

    "Hyderabadd" : "Hyderabad",
    "Hyderabaddd" : "Hyderabad",

    "NewDelhee" : "New Delhi",
    "NewDelhi" : "New Delhi"
}

allowed = ["Bengaluru", "Hyderabad", "New Delhi"]
df_silver = df_silver.replace(city_mapping, subset=["city"])\
                        .withColumn("city", F.when(F.col("city").isNull(),None)
                                            .when(F.col("city").isin(allowed), F.col("city"))
                                            .otherwise(None))
# Now i will save the data in silver table)

In [0]:
df_silver.select("city").distinct().show()

In [0]:
df_silver.select("customer_name").distinct().show() #Few customer name has small latter and few are capital latter , i will fix that

In [0]:
df_silver = df_silver.withColumn(
                         "customer_name" , F.when(F.col("customer_name").isNull(), None)
                        .otherwise(F.initcap(F.col("customer_name") # when you use initcap it will make first latter capital and rest small latter
                     )))

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
# we are checking the null value in city column and analysis them we try to file with some city 
df_silver.filter(F.col("city").isNull()).show(truncate= True)

In [0]:
# this are the customer name has the null value , we are try to fix that
null_customer_name = ['Sprintx Nutrition','Zenathlete Foods','Primefuel Nutrition','Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_name)).show(truncate=False)

In [0]:
# If we niotice that '789403|Sprintx Nutrition|NULL' preveously this customer has city name other , after analysis them business manager deside that we can put other city name 

customer_city_fix = {
    789403 : "New Delhi",

    789420 : "Bengaluru" ,

    789521 : "Hyderabad",

    789603 : "Hyderabad"

}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)


df_silver.createOrReplaceTempView("dfsilver")
df_fix.createOrReplaceTempView("dffix")

df_silver_final = spark.sql("""
    SELECT 
        s.customer_id,
        s.customer_name,
        COALESCE(s.city, f.fixed_city) AS city,
        s.read_timestamp,
        s.file_name,
        s.file_size
    FROM dfsilver s
    LEFT JOIN dffix f
        ON s.customer_id = f.customer_id
""")

In [0]:
s = df_silver.alias("s")
f = df_fix.alias("f")

df_silver_final = (
    s.join(
        f,
        F.col("s.customer_id") == F.col("f.customer_id"),
        "left"
    )
    .select(
        F.col("s.customer_id"),
        F.col("s.customer_name"),
        F.coalesce(
            F.col("s.city"),
            F.col("f.fixed_city")  # Replace null with fixed_city
        ).alias("city"),
        F.col("s.read_timestamp"),
        F.col("s.file_name"),
        F.col("s.file_size")
    )
)

In [0]:
df_silver_final.show()

In [0]:
df_silver_final = df_silver_final.dropna(subset=["city"])
df_silver_final.show()


In [0]:
df_silver_final = df_silver_final.withColumn("customer_id" , F.col("customer_id").cast("string"))
df_silver_final.printSchema()

In [0]:
df_silver_final = df_silver_final.withColumn(
                                        "customer",
                                        F.concat_ws("-", F.col("customer_name"), F.coalesce(F.col("city"), F.lit("Unknown"))
                                    ))
df_silver_final = (df_silver_final.withColumn("market",F.lit("India"))
                                    .withColumn("platform", F.lit("Sports Bar"))
                                    .withColumn("channel", F.lit("Acquisition"))

)                                                  

In [0]:
df_silver_final.show()

In [0]:
# load it to silver table 
df_silver_final.write\
                .format("delta")\
                .option("delta.enableChangeDataFeed", "true")\
                .option("mergeSchema", "true")\
                .mode("overwrite")\
                .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold Processing 

In [0]:
df_silver_for_gold = spark.sql(f"""
                               select * from {catalog}.{silver_schema}.{data_source}
                               """)
df_gold = df_silver_for_gold.select("customer_id","customer_name","city","customer","market","platform","channel")
display(df_gold)

In [0]:
df_gold.write\
                .format("delta")\
                .option("delta.enableChangeDataFeed", "true")\
                .option("mergeSchema","true")\
                .mode("overwrite")\
                .saveAsTable(f"{catalog}.{gold_schema}.sb_dm_{data_source}") # this gold data for sports bar company that's why sb_dm


In [0]:
# For this code merging the parent company data with the child company data in dim_customer table (gold table)
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_chile_customers = spark.table("fmcg.gold.sb_dm_coustmers").select(
                F.col("customer_id").alias("customer_code"),
                "customer",
                "market",
                "platform",
                "channel"
)

In [0]:
# Upsert the data into the delta table
delta_table.alias("target").merge(
    source= df_chile_customers.alias("source"),
    condition = "target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()